# Exercise — Rule-Based IE on OCR-like Invoice Text

In this exercise, you will build a small rule-based Information Extraction pipeline for noisy OCR-like invoice text.

The goal is not to build a perfect system. The goal is to understand how rule-based extraction works, where it helps, and where it breaks.

## Learning Objectives

By the end of this exercise, you should be able to:
- inspect noisy OCR-like text
- design preprocessing rules for OCR errors
- extract invoice fields using regular expressions
- normalize European number formats
- analyze the limitations of rule-based IE

---

## Scenario

You are given OCR-like text extracted from an invoice. The text contains typical OCR errors, such as:

- `O` instead of `0`
- `S` instead of `5`
- broken or inconsistent spacing
- European number formats such as `1.136,45 EUR`
- German invoice labels such as `Rechnung Nr.`, `Datum`, and `Gesamtbetrag`

Your task is to extract:

- `invoice_number`
- `date`
- `total_amount`

Expected output format:

```python
{
    "invoice_number": "INV-2024-007",
    "date": "05.02.2024",
    "total_amount": 1136.45
}
```

---

In [ ]:
import re
from pprint import pprint

## 1. OCR-like Input Text

The text below simulates OCR output from an invoice. It is not perfectly clean. Inspect it carefully before writing extraction rules.

In [ ]:
ocr_invoice_text = """
RECHNUNG

Muster GmbH
Hauptstrasse 12
21335 Luneburg

Rechnung Nr.: INV-2O24-OO7
Datum: 0S.02.2024
Kunden-Nr: K-8831

Beschreibung              Menge    Preis      Gesamt
Beratung KI-Systeme          2      350,00     700,00
OCR Setup                    1      120,00     12O,00
Support                      3       45,00     135,OO

Zwischensumme:                         955,00 EUR
MwSt 19%:                              181,45 EUR
Gesamtbetrag:                       1.136,45 EUR

Bitte zahlen Sie den Betrag bis zum 19.O2.2024.
IBAN: DE89 3704 0044 0532 O130 00

Vielen Dank fur Ihren Auftrag.
"""

print(ocr_invoice_text)

## Task 1 — Inspect the Text

Before writing code, inspect the text manually.

### Questions

1. Which OCR errors do you see?
2. Which parts look easy to extract?
3. Which parts could be dangerous to clean automatically?
4. Which fields are explicitly labeled?

### Your Notes

- OCR errors:
- Easy fields:
- Dangerous cleaning decisions:
- Explicit labels:

## 2. Preprocessing OCR Text

OCR text often contains character confusions. However, blindly replacing characters can introduce new errors.

For example, replacing every `O` with `0` may fix `INV-2O24-OO7`, but it may damage real words such as `OCR`, `Support`, or company names.

In this task, start with a simple preprocessing function and improve it carefully.

In [ ]:
def preprocess_ocr_text(text: str) -> str:
    """Basic preprocessing for OCR-like invoice text."""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


cleaned_text = preprocess_ocr_text(ocr_invoice_text)
print(cleaned_text)

## Task 2 — Improve Preprocessing

Improve the preprocessing function so it fixes some OCR errors without being too aggressive.

### Suggestions

- Fix `2O24` → `2024`
- Fix `OO7` → `007`
- Fix `0S.02.2024` → `05.02.2024`
- Fix numeric amounts such as `12O,00` and `135,OO`

### Important

Avoid replacing all `O` characters globally unless you can explain why this is risky.

In [ ]:
def improved_preprocess_ocr_text(text: str) -> str:
    """Improve OCR text while trying not to damage normal words."""
    text = re.sub(r"\s+", " ", text)
    
    # TODO: Add careful OCR-specific fixes here.
    # Example ideas:
    # text = re.sub(...)
    
    return text.strip()


improved_text = improved_preprocess_ocr_text(ocr_invoice_text)
print(improved_text)

## 3. Extract the Invoice Number

The invoice number appears after `Rechnung Nr.`.

Because the OCR output contains character errors, extraction may fail unless preprocessing handles them.

In [ ]:
def extract_invoice_number(text: str) -> str | None:
    pattern = r"Rechnung\s*Nr\.?\s*:?\s*([A-Z0-9-]+)"
    match = re.search(pattern, text, flags=re.IGNORECASE)
    return match.group(1) if match else None


invoice_number = extract_invoice_number(improved_text)
print(invoice_number)

## 4. Extract the Invoice Date

The invoice date appears after `Datum:`.

The OCR output contains `0S.02.2024`, where `S` should probably be `5`.

In [ ]:
def extract_date(text: str) -> str | None:
    pattern = r"Datum\s*:\s*(\d{2}\.\d{2}\.\d{4})"
    match = re.search(pattern, text, flags=re.IGNORECASE)
    return match.group(1) if match else None


date = extract_date(improved_text)
print(date)

## 5. Extract the Total Amount

The total amount appears after `Gesamtbetrag:`.

The amount uses a European number format:

```text
1.136,45 EUR
```

Here, the dot is a thousands separator and the comma is a decimal separator.

In [ ]:
def extract_total_amount(text: str) -> str | None:
    pattern = r"Gesamtbetrag\s*:\s*([\d\.,]+)\s*EUR"
    match = re.search(pattern, text, flags=re.IGNORECASE)
    return match.group(1) if match else None


amount = extract_total_amount(improved_text)
print(amount)

## 6. Normalize the Amount

Convert the extracted amount string into a Python `float`.

Example:

```text
1.136,45 → 1136.45
```

In [ ]:
def normalize_euro_amount(amount: str | None) -> float | None:
    if amount is None:
        return None
    
    amount = amount.replace(".", "")
    amount = amount.replace(",", ".")
    
    try:
        return float(amount)
    except ValueError:
        return None


normalized_amount = normalize_euro_amount(amount)
print(normalized_amount)

## 7. Build the Full Rule-Based IE Pipeline

Now combine all steps into one pipeline:

```text
OCR-like text → preprocessing → regex extraction → normalization → structured output
```

In [ ]:
def extract_invoice_fields(text: str) -> dict:
    cleaned = improved_preprocess_ocr_text(text)
    amount = extract_total_amount(cleaned)
    
    return {
        "invoice_number": extract_invoice_number(cleaned),
        "date": extract_date(cleaned),
        "total_amount": normalize_euro_amount(amount),
    }


result = extract_invoice_fields(ocr_invoice_text)
pprint(result)

## 8. Evaluate the Result

Compare your output with the expected result.

In [ ]:
expected = {
    "invoice_number": "INV-2024-007",
    "date": "05.02.2024",
    "total_amount": 1136.45,
}

prediction = extract_invoice_fields(ocr_invoice_text)

print("Prediction:")
pprint(prediction)

print("\nExpected:")
pprint(expected)

print("\nField-level evaluation:")
for field in expected:
    print(field, prediction.get(field) == expected.get(field))

## 9. Robustness Challenge

Now test your pipeline on a second OCR-like invoice text.

This text uses English labels and different OCR errors.

### Task

Adapt your pipeline so that it also works for this example.

Do not hard-code the expected output.

In [ ]:
ocr_invoice_text_2 = """
INVOICE

Bright Data Services Ltd.
Market Street 8
London

Invoice N0: AB-2O24-19
Date: 2024-O3-12
Customer ID: C-202

Item                         Qty     Price       Total
Data cleaning                 4      250.00      1,000.00
OCR evaluation                2      320.45        640.90
Pipeline support              1      840.00        840.00

TotaI Amount Due: EUR 2,48O.9O

Thank y0u for y0ur business.
"""

print(ocr_invoice_text_2)

### Challenge Questions

1. Which parts of your current pipeline fail on the second text?
2. Do the German-specific patterns still work?
3. How can you support both German and English invoice labels?
4. How can you safely handle OCR errors such as `TotaI` instead of `Total`?
5. What becomes hard to maintain as the number of formats increases?

In [ ]:
# Try your current pipeline on the second text.
# Then modify your functions above to make the pipeline more robust.

pprint(extract_invoice_fields(ocr_invoice_text_2))

## 10. Reflection

Answer briefly:

1. Which OCR errors were easiest to fix?
2. Which OCR errors were dangerous to fix with simple replacement rules?
3. Which field was easiest to extract?
4. Which field was hardest to extract?
5. Why do rule-based IE systems become brittle?
6. At what point would you consider using ML or LLM-based extraction instead?


### Your Answers

1. Easiest OCR errors:
2. Dangerous fixes:
3. Easiest field:
4. Hardest field:
5. Why brittle:
6. When to use ML/LLMs:

## Summary

In this exercise, you built a rule-based IE pipeline for OCR-like invoice text.

You practiced:

- OCR-aware preprocessing
- regex-based extraction
- amount normalization
- structured output generation
- field-level evaluation
- error analysis

Main takeaway:

> Rule-based IE can work well in controlled settings, but it becomes brittle as formats, languages, and OCR errors vary.
